# ENG 6010 Test 2, Q1: crime-scene zone classifier (Colab)

The Go2's front-camera frame is used to predict which of 5 crime-scene zones the dog is in (house 185):
`victim_area`, `bed_area`, `lounge_area`, `desk_entry_area` and `bathroom`.

The models are:
- **ResNet18** pretrained on ImageNet and fine-tuned;
- a **small CNN** trained from scratch;
- **colour-histogram + logistic-regression** baselines, in RGB and in HSV.

The data comes from the Q2 simulation and is split **by run**, so the test routes and the held-out scene are never seen in training.

Runtime: GPU recommended (Runtime > Change runtime type > T4/A100). Everything also runs on CPU, just more slowly.

In [ ]:
# 1. Get the code (main branch) and install the dependencies
BRANCH = 'main'
!test -d /content/Unitree-Dog-Claude-Dog- || git clone -q --branch $BRANCH https://github.com/CDJ7/Unitree-Dog-Claude-Dog-.git /content/Unitree-Dog-Claude-Dog-
%cd /content/Unitree-Dog-Claude-Dog-/Test2
!pip -q install -r requirements.txt
import torch; print(torch.__version__, 'CUDA' if torch.cuda.is_available() else 'CPU')

## 2. Data

Choose **one** option:
- **A (full results):** you have `data_full.zip` (a zip of `Test2/data/full/`, made with `cd Test2/data && zip -qr data_full.zip full`)
  on Google Drive. Mount the drive and unzip it.
- **B (full results, no zip):** regenerate the full dataset in Colab (36 runs, about 17k frames) with the MuJoCo simulator.
  This took about 1.5 h with 4 workers here; on a 2-core Colab runtime expect longer.
- **C (quick check, default):** use the 3 committed sample runs (`data/sample`, one each for train / val / test) with
  `config/q1_sample.yaml` (1 epoch). Every cell runs in a few minutes, but **the numbers are not meaningful**; results go to
  `results/q1_sample/`.

In [ ]:
DATA_OPTION = 'C'   # 'A', 'B' or 'C'
import os
if DATA_OPTION == 'A':
    from google.colab import drive; drive.mount('/content/drive')
    !mkdir -p data && unzip -q -o /content/drive/MyDrive/data_full.zip -d data/
elif DATA_OPTION == 'B':
    !git -C .. submodule update --init --depth 1 Phase_2_Investigator/third_party/robot-unitree-go2-mujoco
    !apt-get -qq install -y libegl1 libegl-mesa0 libosmesa6 libgl1-mesa-dri > /dev/null
    os.environ['MUJOCO_GL'] = 'egl'
    !python q2_data_simulation/collect_dataset.py --workers 2
else:
    os.environ['Q1_CONFIG'] = 'config/q1_sample.yaml'     # every Q1 script below reads this
import sys; sys.path.insert(0, '.')
from q1_supervised.data import load_cfg
CFG = load_cfg(); DATA, RESULTS = CFG['data_dir'], CFG['results_dir']
print('config:', os.environ.get('Q1_CONFIG', 'config/q1_train.yaml'), '| data:', DATA, '| results:', RESULTS)
!ls $DATA | head -40

## 3. Baselines: colour histograms (RGB vs HSV) + logistic regression

In [ ]:
!python q1_supervised/baselines.py

## 4. Train ResNet18 (transfer learning) and the small CNN (from scratch)
Hyper-parameters are in `config/q1_train.yaml`; seeds are fixed. On Colab the ImageNet weights come straight from torchvision.

In [ ]:
!python q1_supervised/train.py --model resnet18

In [ ]:
!python q1_supervised/train.py --model smallcnn

## 5. Evaluate on the held-out test runs, Grad-CAM, demo

In [ ]:
!python q1_supervised/evaluate.py
!python q1_supervised/gradcam.py
!python q1_supervised/demo.py          # recorded demo on the first test run

In [ ]:
from IPython.display import Image, Markdown, display
display(Markdown(open(f'{RESULTS}/comparison.md').read()))
import glob
for f in ['curves_resnet18.png', 'confusion_resnet18.png', 'confusion_hist_hsv.png', 'gradcam.png', 'boundary.png'] + \
         sorted(os.path.basename(g) for g in glob.glob(f'{RESULTS}/demo_*.png')):
    display(Image(f'{RESULTS}/{f}'))

## 6. Save the results to Drive (optional)

In [ ]:
from google.colab import drive; drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/test2_results && cp -r $RESULTS /content/drive/MyDrive/test2_results/